# Solutions — Notebook 02: Data Lake

**BINFX410 — Chapter 10**

> **Prerequisite:** Run `02_data_lake.ipynb` first (which requires `01_introduction_and_setup.ipynb`). This builds the Bronze/Silver/Gold Parquet files that the solutions below query.

In [ ]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Connect DuckDB — queries will reference Silver/Gold Parquet files via relative paths
con = duckdb.connect()

print('Setup complete.')
print('Silver and Gold Parquet files expected at ../data_lake/silver/ and ../data_lake/gold/')

## Exercise 2.1 — DuckDB SQL on Parquet

Using DuckDB and the Silver Parquet files, write a query that finds the **top 3 tissue types by total variant count** from PASS calls only. Your output should have columns: `tissue_type`, `variant_count`, `unique_samples`.

Hint: Join `variants` → `variant_calls` → `samples` to link variants to tissue type. Filter where `filter_status = 'PASS'`.

In [ ]:
result_ex1 = con.execute("""
    SELECT
        s.tissue_type,
        COUNT(v.variant_id)          AS variant_count,
        COUNT(DISTINCT s.sample_id)  AS unique_samples
    FROM read_parquet('../data_lake/silver/variants.parquet')      v
    JOIN read_parquet('../data_lake/silver/variant_calls.parquet') vc
      ON v.call_id = vc.call_id
    JOIN read_parquet('../data_lake/silver/samples.parquet')       s
      ON vc.sample_id = s.sample_id
    WHERE vc.filter_status = 'PASS'
    GROUP BY s.tissue_type
    ORDER BY variant_count DESC
    LIMIT 3
""").df()

print('=== Top 3 Tissue Types by PASS Variant Count ===')
print(result_ex1)

## Exercise 2.2 — Build a New Gold Table

Create a new Gold Parquet file `../data_lake/gold/consequence_monthly_counts.parquet` that shows:
- `consequence` (variant consequence type)
- `call_year`
- `call_month`
- `variant_count`
- `unique_samples`

Include only PASS calls. Sort by `consequence`, `call_year`, `call_month`.

In [ ]:
# Read silver layers
df_variants_s = pd.read_parquet('../data_lake/silver/variants.parquet')
df_calls_s    = pd.read_parquet('../data_lake/silver/variant_calls.parquet')

# PASS calls only
pass_calls = df_calls_s[df_calls_s['filter_status'] == 'PASS'][['call_id', 'call_date', 'call_year', 'call_month']]

# Join variants to get call date
variants_dated = df_variants_s.merge(pass_calls, on='call_id', how='inner')

# Aggregate
df_consequence_monthly = (
    variants_dated
    .groupby(['consequence', 'call_year', 'call_month'])
    .agg(
        variant_count=('variant_id', 'count'),
        unique_samples=('call_id', 'nunique'),
    )
    .reset_index()
    .sort_values(['consequence', 'call_year', 'call_month'])
)

os.makedirs('../data_lake/gold', exist_ok=True)
pq.write_table(
    pa.Table.from_pandas(df_consequence_monthly, preserve_index=False),
    '../data_lake/gold/consequence_monthly_counts.parquet',
    compression='snappy'
)
print(f'Written: consequence_monthly_counts.parquet — {len(df_consequence_monthly)} rows')
print(df_consequence_monthly.head(10))

## Exercise 2.3 — Schema Evolution Analysis

The two files in `../data_lake/demo_drift/` have incompatible schemas (`variants_batch1.parquet` vs `variants_batch2.parquet`).

**a)** Write a Python function `check_schema_compatibility(path1, path2)` that returns `True` if both files have the same column names and types, `False` otherwise, and prints a diff of any differences.

In [ ]:
def check_schema_compatibility(path1: str, path2: str) -> bool:
    """Return True if schemas are identical, False otherwise. Prints a diff of differences."""
    schema1 = pq.read_schema(path1)
    schema2 = pq.read_schema(path2)

    fields1 = {f.name: str(f.type) for f in schema1}
    fields2 = {f.name: str(f.type) for f in schema2}

    cols_only_in_1 = set(fields1) - set(fields2)
    cols_only_in_2 = set(fields2) - set(fields1)
    type_mismatches = {
        col: (fields1[col], fields2[col])
        for col in set(fields1) & set(fields2)
        if fields1[col] != fields2[col]
    }

    compatible = not cols_only_in_1 and not cols_only_in_2 and not type_mismatches

    if compatible:
        print('Schemas are compatible.')
    else:
        print('Schemas are INCOMPATIBLE:')
        for col in sorted(cols_only_in_1):
            print(f'  Only in {path1}: {col} ({fields1[col]})')
        for col in sorted(cols_only_in_2):
            print(f'  Only in {path2}: {col} ({fields2[col]})')
        for col, (t1, t2) in sorted(type_mismatches.items()):
            print(f'  Type mismatch on "{col}": {t1} vs {t2}')

    return compatible


# Test the function on the demo drift files
check_schema_compatibility(
    '../data_lake/demo_drift/variants_batch1.parquet',
    '../data_lake/demo_drift/variants_batch2.parquet'
)

## Exercise 2.3b — Automatic Schema Change Detection

**How would you design a Bronze → Silver pipeline that detects schema changes automatically and alerts the bioinformatics team before writing to Silver?**

---

A Bronze → Silver pipeline can detect schema changes by:

1. **Store the expected schema as metadata.** On first load of each table, serialize the PyArrow schema to JSON and save it to a `./data_lake/schemas/<table_name>.json` file (or a database).

2. **Compare on every ingest run.** Before writing a new Bronze file, call `check_schema_compatibility()` against the stored expected schema. If it returns `False`, halt the pipeline and raise an alert.

3. **Alert mechanism.** Send a notification (email, Slack webhook, PagerDuty) with the diff output so the data engineering team can evaluate whether the change is intentional.

4. **Explicit schema evolution.** If the change is approved, run `ALTER TABLE` (or re-register the schema) and update the stored metadata. This creates an audit trail of intentional schema changes.

This is exactly what Delta Lake does automatically with schema enforcement — the transaction log stores the canonical schema, and every write is validated against it.